# 01 — Phase 1 LoRA retrain (Drive-checkpointed)
Clones pes-mtech-project, trains the paper-closer LoRA config (r=16, alpha=128, 5 epochs, 4000 samples), and streams every checkpoint + the best adapter to Google Drive so a disconnect never loses the best params.

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf /content/repo
!git clone https://github.com/kvarun314/pes-mtech-project.git /content/repo
%cd /content/repo
!pip install -q -e ".[colab]"
!pip install -q kaggle

In [ ]:
import os
from huggingface_hub import login

# Colab Secrets panel (recommended): add a secret named HF_TOKEN, then it
# reaches this cell via google.colab.userdata — NOT via os.environ.
# Accept the Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B
# before this token will work.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")
login(hf_token)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RUNS_DIR = "/content/drive/MyDrive/pes-mtech-project/runs"

## Download the Kaggle dataset
Get `kaggle.json` from https://www.kaggle.com/settings -> Create New Token,
then upload it with the file-upload button in the Colab sidebar (it lands in `/content/`).
The next cell moves it into place and downloads the dataset.

In [ ]:
import os, shutil

os.makedirs("/root/.kaggle", exist_ok=True)
if os.path.exists("/content/kaggle.json"):
    shutil.move("/content/kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d arhamrumi/amazon-product-reviews -p /content/data --unzip
DATA_PATH = "/content/data/Reviews.csv"

In [ ]:
from agentic_sentiment.train.run_dir import RunDir
from agentic_sentiment.train.phase1_train import run_training

run_dir = RunDir(base_dir=DRIVE_RUNS_DIR)
print("Run directory:", run_dir.path)
metrics = run_training(DATA_PATH, run_dir)
print(metrics)